# Deep-Value Reclaim Screen — furthest under the 200-SMA, back above the 20-SMA
**The question (Jake, 2026-07-25):** what happens if you buy the S&P names FURTHEST below their 200-day SMA that have
reclaimed their 20-day SMA (the trend-turn "catalyst")?

**Token-free**: pulls Yahoo v8 chart data directly (no key). Run top to bottom. ~3-4 min for the download cell.

## ⚠️ Read before believing the numbers
1. **SURVIVORSHIP BIAS — the big one, and it flatters THIS strategy specifically.** The universe is TODAY'S S&P 500.
   Every deeply-beaten-down pick in the backtest is a company we already know survived and stayed in the index. In real
   time, some furthest-under-200 names get deleted, acquired at lows, or go to zero — those losses are invisible here.
   Real results would be meaningfully worse. Direction of every bias in this notebook: UP.
2. No transaction costs, slippage, or taxes (10-name monthly rebalance = high turnover).
3. Adjusted closes (dividends included via Yahoo adjclose).
4. First test run 2026-07-25 (10y data): strategy 21.0% CAGR / −45.8% maxDD / Sharpe 0.86 vs SPY 15.0% / −23.9% / 0.95.
   Higher return, WORSE risk-adjusted; wins concentrated in recovery years (2020, 2023), crushed in 2022 (−34%).

In [ ]:
# CELL 1 — parameters
RANGE      = '10y'   # history to pull
N          = 10      # names per basket
FILTER_20  = True    # require close >= 20-day SMA (the reclaim 'catalyst')
REBAL      = 'M'     # 'M' monthly (last trading day) or 'W' weekly (Fridays)
SLEEP      = 0.12    # be polite to Yahoo

import requests, time, re, os, json
import pandas as pd, numpy as np
S = requests.Session(); S.headers['User-Agent'] = 'Mozilla/5.0 (research notebook)'
print('ready')

In [ ]:
# CELL 2 — current S&P 500 constituents from Wikipedia
html = S.get('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies', timeout=30).text
raw = sorted(set(re.findall(r'href="https?://(?:www\.)?(?:nyse|nasdaq|cboe)\.com[^"]*"[^>]*>([A-Z][A-Z0-9.\-]*)<', html)))
TICKERS = [t.replace('.', '-') for t in raw]   # Yahoo uses '-' for class shares (BRK.B -> BRK-B)
if len(TICKERS) < 400:   # fallback parser if the page layout changed
    tables = pd.read_html(html)
    TICKERS = [t.replace('.', '-') for t in tables[0]['Symbol'].astype(str).tolist()]
print(len(TICKERS), 'tickers')

In [ ]:
# CELL 3 — download 10y daily adjusted closes (Yahoo v8, no key). ~3-4 min.
def pull(t):
    r = S.get(f'https://query1.finance.yahoo.com/v8/finance/chart/{t}',
              params={'range': RANGE, 'interval': '1d', 'events': 'div,split'}, timeout=20)
    j = r.json()['chart']['result'][0]
    adj = j['indicators'].get('adjclose', [{}])[0].get('adjclose') or j['indicators']['quote'][0]['close']
    s = pd.Series(adj, index=pd.to_datetime(j['timestamp'], unit='s').normalize())
    return s[~s.index.duplicated()]

cols, failed = {}, []
for i, t in enumerate(TICKERS + ['SPY']):
    try:
        cols[t] = pull(t)
    except Exception:
        failed.append(t)
    if i % 50 == 0: print(i, end=' ', flush=True)
    time.sleep(SLEEP)
print()
px = pd.DataFrame(cols).sort_index()
spy = px.pop('SPY')
print('matrix:', px.shape, px.index[0].date(), '->', px.index[-1].date(), '| failed:', failed)

In [ ]:
# CELL 4 — backtest
sma200 = px.rolling(200, min_periods=200).mean()
sma20  = px.rolling(20,  min_periods=20).mean()
dist   = px / sma200 - 1.0

period = px.index.to_period(REBAL)
rebal_dates = pd.DatetimeIndex(px.groupby(period).apply(lambda g: g.index[-1]).values)
rebal_dates = rebal_dates[rebal_dates >= px.index[210]]   # need 200d of history

def run(filter20=FILTER_20, n=N):
    rets, picks = {}, []
    for i in range(len(rebal_dates) - 1):
        d0, d1 = rebal_dates[i], rebal_dates[i + 1]
        e = dist.loc[d0].dropna()
        e = e[e < 0]                                   # must be UNDER the 200-SMA
        if filter20:
            ok = px.loc[d0] >= sma20.loc[d0]           # the reclaim 'catalyst'
            e = e[ok.reindex(e.index).fillna(False)]
        sel = e.sort_values().head(n).index            # deepest under
        rets[d1] = (px.loc[d1, sel] / px.loc[d0, sel] - 1).mean() if len(sel) else 0.0
        picks.append((d0, list(sel)))
    return pd.Series(rets), picks

def stats(s, label, ppyr):
    curve = (1 + s).cumprod()
    yrs = (s.index[-1] - s.index[0]).days / 365.25
    cagr = curve.iloc[-1] ** (1 / yrs) - 1
    dd = (curve / curve.cummax() - 1).min()
    vol = s.std() * np.sqrt(ppyr)
    print(f'{label:42s} CAGR {cagr*100:6.1f}%  maxDD {dd*100:6.1f}%  vol {vol*100:5.1f}%  Sharpe {(s.mean()*ppyr)/vol if vol>0 else 0:5.2f}')
    return curve

ppyr = 12 if REBAL == 'M' else 52
strat, picks_log = run(FILTER_20, N)
nofilt, _        = run(False, N)
spy_r = spy.reindex(rebal_dates).pct_change().dropna().reindex(strat.index).fillna(0)

print(f'=== {REBAL} rebalance since {strat.index[0].date()} ===')
c1 = stats(strat,  f'Deepest-under-200 + above-20 (N={N})', ppyr)
c2 = stats(nofilt, f'Deepest-under-200, NO 20d filter (N={N})', ppyr)
c3 = stats(spy_r,  'SPY buy-hold (same dates)', ppyr)
print(f'\nBeats SPY in {(strat > spy_r).mean()*100:.0f}% of periods; corr {strat.corr(spy_r):.2f}')
print(f'Strategy: mean {strat.mean()*100:.2f}%  median {strat.median()*100:.2f}%  best {strat.max()*100:.1f}%  worst {strat.min()*100:.1f}%')

byyr = pd.DataFrame({'strategy': strat, 'SPY': spy_r}).groupby(strat.index.year).apply(lambda g: (1 + g).prod() - 1)
print('\nBy year (%):'); print((byyr * 100).round(1).to_string())

In [ ]:
# CELL 5 — TODAY'S screen (what the strategy would buy right now)
d0 = px.index[-1]
e = dist.loc[d0].dropna(); e = e[e < 0]
ok = px.loc[d0] >= sma20.loc[d0]
e_f = e[ok.reindex(e.index).fillna(False)]
today = e_f.sort_values().head(N)
print(f'As of {d0.date()} — deepest under 200-SMA AND at/above 20-SMA:')
for t, d in today.items():
    print(f'  {t:6s} {d*100:7.1f}% below its 200-SMA   (close {px.loc[d0, t]:.2f})')
print(f'\n(under-200 universe: {len(e)} names; passing the 20-SMA reclaim: {len(e_f)})')

In [ ]:
# CELL 6 — equity curves
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 5))
c1.plot(ax=ax, label=f'Deep-value reclaim (N={N})')
c2.plot(ax=ax, label='No 20-SMA filter')
c3.plot(ax=ax, label='SPY')
ax.set_yscale('log'); ax.legend(); ax.set_title('Growth of $1 — deep-value reclaim vs SPY (survivorship-flattered)')
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Study 2 — DRIFT vs GAP declines (Jake's "no external catalyst" question, 2026-07-25)
Hypothesis tested: a stock in a bear market (≥20% off its 252-day high) with NO news catalyst + a 20-SMA reclaim = hidden value.
Catalyst proxy from price shape: worst MARKET-ADJUSTED single day (stock minus SPY) since the 252d high. ≤ −8% = GAP (idiosyncratic
catalyst); otherwise DRIFT (slow, no-news bleed).

**First run result (2026-07-25): the hypothesis INVERTED.** GAP decliners: 29.2% CAGR / Sharpe 1.07. DRIFT decliners: 13.4% CAGR /
Sharpe 0.65 (UNDER SPY's 15.0%/0.94). Robust to 50-SMA, −6% threshold, raw vs market-adjusted. Read: quiet declines are informed,
persistent selling (no press release ≠ no reason); loud single-day crushes are where overreaction — and the recovery edge — lives.
Same survivorship caveat as Study 1 (both cohorts share it, so the COMPARISON is cleaner than either absolute number).

In [ ]:
# CELL 7 — Study 2: DRIFT (no-catalyst) vs GAP (catalyst) bear-market decliners, 20-SMA reclaim
SHOCK = -0.08     # worst market-adjusted day threshold: <= this = 'catalyst' (GAP cohort)

ret1 = px.pct_change()
xret = ret1.sub(spy.pct_change(), axis=0)          # market-adjusted daily returns
high252 = px.rolling(252, min_periods=252).max()
dd252 = px / high252 - 1.0

def run_study2(shock=SHOCK, n=N):
    drift_r, gap_r, nd, ng = {}, {}, [], []
    for i in range(len(rebal_dates) - 1):
        d0, d1 = rebal_dates[i], rebal_dates[i + 1]
        row = dd252.loc[d0].dropna()
        bears = row[row <= -0.20]                   # in a bear market vs 1y high
        ok = px.loc[d0] >= sma20.loc[d0]            # the reclaim
        bears = bears[ok.reindex(bears.index).fillna(False)]
        drift, gap = [], []
        for t in bears.index:
            win = px[t].loc[:d0].iloc[-252:]
            worst = xret[t].loc[win.idxmax():d0].min()   # worst idiosyncratic day since the high
            (gap if worst <= shock else drift).append(t)
        nd.append(len(drift)); ng.append(len(gap))
        for cohort, store in [(drift, drift_r), (gap, gap_r)]:
            sel = row[cohort].sort_values().head(n).index
            store[d1] = (px.loc[d1, sel] / px.loc[d0, sel] - 1).mean() if len(sel) else float('nan')
    return pd.Series(drift_r), pd.Series(gap_r), np.mean(nd), np.mean(ng)

def stats2(s, label):
    s2 = s.dropna(); curve = (1 + s.fillna(0)).cumprod()
    yrs = (s2.index[-1] - s2.index[0]).days / 365.25
    cagr = curve.iloc[-1] ** (1 / yrs) - 1
    mdd = (curve / curve.cummax() - 1).min()
    vol = s2.std() * np.sqrt(12); sh = (s2.mean() * 12) / vol if vol > 0 else 0
    print(f'{label:40s} CAGR {cagr*100:6.1f}%  maxDD {mdd*100:6.1f}%  Sharpe {sh:5.2f}  n_mo {len(s2)}')

d_s, g_s, ndm, ngm = run_study2()
print(f'avg eligible per month: DRIFT {ndm:.0f}, GAP {ngm:.0f}')
stats2(d_s, 'DRIFT decliners (no-catalyst proxy)')
stats2(g_s, 'GAP decliners (idiosyncratic catalyst)')
stats2(spy_r, 'SPY')
byyr = pd.DataFrame({'gap': g_s, 'drift': d_s}).groupby(g_s.index.year).apply(lambda x: (1 + x.fillna(0)).prod() - 1)
print(); print((byyr * 100).round(1).to_string())